# Laboratorio 4 – Árboles de Decisión
## CC3074 – Minería de Datos | UVG Semestre I 2026

**Universidad del Valle de Guatemala**  
Facultad de Ingeniería - Departamento de Ciencias de la Computación

Jonathan Díaz - 23837

Martín Pérez - 23234

Karen Toledo - 23692

SmartStay Advisors necesita estimar precios competitivos de propiedades Airbnb. En esta entrega construimos modelos de **árboles de decisión** (regresión y clasificación) y los comparamos con los modelos de regresión lineal del Lab 3.

## 0. Importación de librerías

In [ ]:
import pyreadr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
RANDOM_STATE = 42

print('Librerías cargadas correctamente')

## 1. Carga del Dataset

El dataset proviene de listados de Airbnb y está en formato `.RData`. Contiene información de propiedades: características físicas, disponibilidad, reseñas y precio. Lo cargamos con `pyreadr` y hacemos una inspección inicial para entender su estructura.

In [ ]:
result = pyreadr.read_r('listings(1).RData')
df = result[list(result.keys())[0]]

print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
df.head(3)

In [ ]:
print('Tipos de datos:')
print(df.dtypes.value_counts())
print()

nulls = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
nulls = nulls[nulls > 0]
print(f'Columnas con valores nulos ({len(nulls)} columnas):')
print(nulls.round(1).to_string())

## 2. Análisis Exploratorio de Datos (EDA)

El análisis exploratorio nos permite entender la distribución de las variables, detectar outliers y descubrir relaciones entre las características y la variable objetivo (`price`). Organizamos el EDA en las siguientes etapas:

1. Limpieza de la variable objetivo
2. Distribución del precio
3. Variables numéricas y su correlación con el precio
4. Variables categóricas relevantes
5. Preprocesamiento

### 2.1 Limpieza de la variable objetivo: `price`

En datasets de Airbnb el precio suele venir como string con formato `"$120.00"`. Debemos convertirlo a numérico antes de cualquier análisis.

In [ ]:
print('Tipo original:', df['price'].dtype)
print('Muestra:', df['price'].dropna().head(5).tolist())

In [ ]:
df['price'] = df['price'].str.replace(r'[\$,]', '', regex=True)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price']).copy()
print(f'Registros con precio válido: {len(df):,}')
print(df['price'].describe())

### 2.2 Distribución del precio

Analizamos la distribución de `price` para detectar sesgo y outliers. Una distribución muy sesgada podría afectar los modelos de regresión.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución del Precio (original)')
axes[0].set_xlabel('Precio (USD)')
axes[0].set_ylabel('Frecuencia')

axes[1].boxplot(df['price'].clip(upper=df['price'].quantile(0.99)),
                vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Boxplot del Precio (sin outliers extremos)')
axes[1].set_xlabel('Precio (USD)')

plt.tight_layout()
plt.show()

print(f'Sesgo: {df["price"].skew():.2f}')
print(f'Curtosis: {df["price"].kurt():.2f}')
print(f'Precio mínimo: ${df["price"].min():.2f}')
print(f'Precio máximo: ${df["price"].max():,.2f}')
print(f'Precio mediano: ${df["price"].median():.2f}')

### 2.3 Transformación logarítmica del precio

Dado el fuerte sesgo positivo del precio, aplicamos transformación logarítmica (`log1p`) para aproximar una distribución normal, lo que mejora el comportamiento de los modelos de regresión lineal.

In [ ]:
p99 = df['price'].quantile(0.99)
df = df[(df['price'] > 0) & (df['price'] <= p99)].copy()
print(f'Registros tras filtrar outliers extremos (p99=${p99:.2f}): {len(df):,}')

df['log_price'] = np.log1p(df['price'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['price'], bins=80, color='salmon', edgecolor='white')
axes[0].set_title('Precio (después de filtrar outliers)')
axes[0].set_xlabel('Precio (USD)')

axes[1].hist(df['log_price'], bins=80, color='steelblue', edgecolor='white')
axes[1].set_title('log(Precio + 1) – distribución más normal')
axes[1].set_xlabel('log(Precio)')

plt.tight_layout()
plt.show()

print(f'Sesgo log_price: {df["log_price"].skew():.3f}')

### 2.4 Variables numéricas y correlación con el precio

Calculamos la correlación de Pearson entre las variables numéricas y `log_price` para identificar los mejores predictores candidatos.

In [ ]:
exclude_cols = ['id', 'scrape_id', 'host_id', 'latitude', 'longitude', 'price', 'log_price']
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in exclude_cols]

corr_with_price = df[num_cols + ['log_price']].corr()['log_price'].drop('log_price')
corr_with_price = corr_with_price.sort_values(key=abs, ascending=False)

top20 = corr_with_price.head(20)
plt.figure(figsize=(10, 7))
colors = ['steelblue' if v >= 0 else 'salmon' for v in top20.values]
plt.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 20 variables con mayor correlación con log(precio)')
plt.xlabel('Correlación de Pearson')
plt.tight_layout()
plt.show()

print('Top 10 variables:')
print(corr_with_price.head(10).round(3).to_string())

### 2.5 Variables categóricas relevantes

Exploramos `room_type` y `city` que pueden tener alta influencia en el precio.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rt_mean = df.groupby('room_type')['price'].median().sort_values(ascending=False)
rt_mean.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Precio mediano por tipo de habitación')
axes[0].set_xlabel('Tipo de habitación')
axes[0].set_ylabel('Precio mediano (USD)')
axes[0].tick_params(axis='x', rotation=30)

city_mean = df.groupby('city')['price'].median().sort_values(ascending=False).head(10)
city_mean.plot(kind='bar', ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Precio mediano por ciudad (top 10)')
axes[1].set_xlabel('Ciudad')
axes[1].set_ylabel('Precio mediano (USD)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Distribución room_type:')
print(df['room_type'].value_counts())

### 2.6 Preprocesamiento

Antes de modelar realizamos:
- Eliminar columnas con >50% de nulos
- Imputar medianas en numéricas
- Codificar `room_type` con one-hot encoding

In [ ]:
thresh = 0.5 * len(df)
df_clean = df.dropna(thresh=int(thresh), axis=1).copy()
print(f'Columnas eliminadas por >50% nulos: {df.shape[1] - df_clean.shape[1]}')
print(f'Columnas restantes: {df_clean.shape[1]}')

num_cols_clean = df_clean.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols_clean:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

df_clean = pd.get_dummies(df_clean, columns=['room_type'], drop_first=True, dtype=int)

print('Preprocesamiento completado.')
print(f'Shape final: {df_clean.shape}')

## 3. Análisis de Grupos (Clustering)

Aplicamos **KMeans** sobre las variables numéricas más relevantes para identificar segmentos naturales de propiedades. Usamos el **método del codo** para determinar el número óptimo de clusters.

In [ ]:
cluster_features = ['accommodates', 'bedrooms', 'beds', 'log_price',
                    'number_of_reviews', 'review_scores_rating',
                    'availability_365', 'reviews_per_month']

df_clust = df[cluster_features].dropna().copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clust)

inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, marker='o', color='steelblue')
plt.title('Método del Codo – KMeans')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia')
plt.xticks(K_range)
plt.tight_layout()
plt.show()

In [ ]:
K_OPTIMAL = 4
km_final = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=10)
df_clust['cluster'] = km_final.fit_predict(X_scaled)

plt.figure(figsize=(10, 6))
for c in range(K_OPTIMAL):
    mask = df_clust['cluster'] == c
    plt.scatter(df_clust.loc[mask, 'accommodates'],
                df_clust.loc[mask, 'log_price'],
                alpha=0.3, s=8, label=f'Cluster {c}')
plt.xlabel('Capacidad (accommodates)')
plt.ylabel('log(Precio)')
plt.title('Clusters de propiedades: Capacidad vs Precio')
plt.legend()
plt.tight_layout()
plt.show()

cluster_summary = df_clust.groupby('cluster').mean().round(2)
print(cluster_summary.to_string())

### Interpretación de clusters

Con base en las medias por grupo podemos caracterizar cada cluster:

- **Cluster 0**: Propiedades de bajo precio, baja capacidad – habitaciones privadas/compartidas económicas.
- **Cluster 1**: Propiedades medianas con buenas reseñas y disponibilidad alta – mercado estándar.
- **Cluster 2**: Alta capacidad y precio elevado – casas/apartamentos completos de lujo.
- **Cluster 3**: Propiedades con pocas reseñas y alta disponibilidad – listados nuevos o poco populares.

Esta segmentación confirma que **capacidad, reseñas y disponibilidad** son dimensiones clave que diferencian el precio.

## 4. Ingeniería de Características y División Train/Test

Seleccionamos las features con base en correlación, disponibilidad de datos y dominio del negocio. Dividimos en **70% entrenamiento y 30% prueba** con `random_state=42` — esta misma división se usa en todos los modelos.

In [ ]:
room_dummies = [c for c in df_clean.columns if c.startswith('room_type_')]

feature_candidates = [
    'accommodates', 'bedrooms', 'beds',
    'minimum_nights', 'availability_365',
    'number_of_reviews', 'reviews_per_month',
    'review_scores_rating', 'review_scores_cleanliness',
    'calculated_host_listings_count',
    'number_of_reviews_ltm',
] + room_dummies

feature_candidates = [c for c in feature_candidates if c in df_clean.columns]

df_model = df_clean[feature_candidates + ['log_price', 'price']].dropna().copy()
print(f'Dataset de modelado: {df_model.shape[0]:,} filas x {len(feature_candidates)} features')
print('Features:', feature_candidates)

In [ ]:
# Heatmap de correlación
corr_matrix = df_model.drop(columns='price').corr()

plt.figure(figsize=(12, 9))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlación entre features seleccionadas y log(precio)')
plt.tight_layout()
plt.show()

In [ ]:
X = df_model[feature_candidates]
y = df_model['log_price']
price_raw = df_model['price']

X_train, X_test, y_train, y_test, price_train, price_test = train_test_split(
    X, y, price_raw, test_size=0.3, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]:,} filas ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test:  {X_test.shape[0]:,} filas ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'Features: {X_train.shape[1]}')
print(f'random_state={RANDOM_STATE} – mismos splits en todos los modelos')

## 5. Modelo Univariado de Regresión Lineal

Seleccionamos la variable con **mayor correlación absoluta** con `log_price` y construimos un modelo simple.

In [ ]:
corr_vals = X_train.corrwith(y_train).abs().sort_values(ascending=False)
best_feature = corr_vals.index[0]
print(f'Variable seleccionada: {best_feature}  (r={corr_vals[best_feature]:.3f})')

X_uni_train = X_train[[best_feature]]
X_uni_test  = X_test[[best_feature]]

model_uni = LinearRegression()
model_uni.fit(X_uni_train, y_train)

y_pred_uni_train = model_uni.predict(X_uni_train)
y_pred_uni_test  = model_uni.predict(X_uni_test)

r2_uni_test  = r2_score(y_test, y_pred_uni_test)
rmse_uni     = np.sqrt(mean_squared_error(y_test, y_pred_uni_test))
mae_uni      = mean_absolute_error(y_test, y_pred_uni_test)

print(f'Intercepto:  {model_uni.intercept_:.4f}')
print(f'Coeficiente: {model_uni.coef_[0]:.4f}')
print(f'R2 Train:    {r2_score(y_train, y_pred_uni_train):.4f}')
print(f'R2 Test:     {r2_uni_test:.4f}')
print(f'RMSE Test:   {rmse_uni:.4f}')
print(f'MAE Test:    {mae_uni:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = X_test[[best_feature]].copy()
sample['y_real'] = y_test.values
sample['y_pred'] = y_pred_uni_test
sample = sample.sample(min(2000, len(sample)), random_state=RANDOM_STATE)

axes[0].scatter(sample[best_feature], sample['y_real'], alpha=0.3, s=8, color='steelblue', label='Datos reales')
xs = np.linspace(sample[best_feature].min(), sample[best_feature].max(), 100)
axes[0].plot(xs, model_uni.intercept_ + model_uni.coef_[0]*xs, color='red', linewidth=2, label='Regresión')
axes[0].set_xlabel(best_feature)
axes[0].set_ylabel('log(Precio)')
axes[0].set_title(f'Regresión univariada: {best_feature}')
axes[0].legend()

residuals_uni = y_test.values - y_pred_uni_test
axes[1].scatter(y_pred_uni_test, residuals_uni, alpha=0.3, s=8, color='salmon')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Valores ajustados')
axes[1].set_ylabel('Residuos')
axes[1].set_title('Residuos vs Valores Ajustados (univariado)')

plt.tight_layout()
plt.show()

### Interpretación

El modelo univariado captura la relación lineal positiva entre la variable seleccionada y el precio. Sin embargo, el R² relativamente bajo indica que **una sola variable no es suficiente** para explicar la variabilidad del precio.

## 6. Regresión Lineal Múltiple

Entrenamos un modelo con todas las features seleccionadas usando `statsmodels` para obtener el resumen completo.

In [ ]:
X_train_f = X_train.astype(float)
X_test_f  = X_test.astype(float)

X_train_sm = sm.add_constant(X_train_f)
X_test_sm  = sm.add_constant(X_test_f)

model_ols = sm.OLS(y_train.astype(float), X_train_sm).fit()
print(model_ols.summary())

In [ ]:
y_pred_multi_train = model_ols.predict(X_train_sm)
y_pred_multi_test  = model_ols.predict(X_test_sm)

r2_multi_train = r2_score(y_train, y_pred_multi_train)
r2_multi_test  = r2_score(y_test,  y_pred_multi_test)
rmse_multi = np.sqrt(mean_squared_error(y_test, y_pred_multi_test))
mae_multi  = mean_absolute_error(y_test, y_pred_multi_test)

print(f'R2 Train:  {r2_multi_train:.4f}')
print(f'R2 Test:   {r2_multi_test:.4f}')
print(f'RMSE Test: {rmse_multi:.4f}')
print(f'MAE Test:  {mae_multi:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_multi_test, alpha=0.2, s=8, color='steelblue')
mn, mx = float(y_test.min()), float(y_test.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Valores reales (log precio)')
axes[0].set_ylabel('Valores predichos')
axes[0].set_title('Real vs Predicho – Modelo Múltiple')
axes[0].legend()

res_multi = y_test.values - y_pred_multi_test
axes[1].scatter(y_pred_multi_test, res_multi, alpha=0.2, s=8, color='salmon')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Valores ajustados')
axes[1].set_ylabel('Residuos')
axes[1].set_title('Residuos vs Ajustados – Modelo Múltiple')

plt.tight_layout()
plt.show()

## 7. Análisis de Multicolinealidad y Modelo Refinado

Calculamos el **VIF** para detectar multicolinealidad y luego eliminamos variables problemáticas.

In [ ]:
X_vif = sm.add_constant(X_train.astype(float))
vif_data = pd.DataFrame()
vif_data['Feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data[vif_data['Feature'] != 'const'].sort_values('VIF', ascending=False)

print('=== VIF por variable ===')
print(vif_data.to_string(index=False))

plt.figure(figsize=(10, 5))
colors_vif = ['red' if v > 10 else 'orange' if v > 5 else 'steelblue' for v in vif_data['VIF']]
plt.barh(vif_data['Feature'][::-1], vif_data['VIF'][::-1], color=colors_vif[::-1])
plt.axvline(5,  color='orange', linestyle='--', linewidth=1, label='VIF=5')
plt.axvline(10, color='red',    linestyle='--', linewidth=1, label='VIF=10')
plt.xlabel('VIF')
plt.title('Variance Inflation Factor por variable')
plt.legend()
plt.tight_layout()
plt.show()

gap = r2_multi_train - r2_multi_test
print(f'\nGap overfitting: {gap:.4f} ({"posible overfitting" if gap > 0.05 else "sin overfitting significativo"})')

In [ ]:
pvals = model_ols.pvalues.drop('const')
high_vif_vars = vif_data[vif_data['VIF'] > 10]['Feature'].tolist()
non_sig_vars  = pvals[pvals > 0.05].index.tolist()
drop_vars = list(set(high_vif_vars + non_sig_vars))
print(f'Variables eliminadas: {drop_vars}')

refined_features = [c for c in feature_candidates if c not in drop_vars]
print(f'Features refinadas ({len(refined_features)}): {refined_features}')

X_train_ref    = X_train[refined_features].astype(float)
X_test_ref     = X_test[refined_features].astype(float)
X_train_ref_sm = sm.add_constant(X_train_ref)
X_test_ref_sm  = sm.add_constant(X_test_ref)

model_ref = sm.OLS(y_train.astype(float), X_train_ref_sm).fit()
print(model_ref.summary())

In [ ]:
y_pred_ref_train = model_ref.predict(X_train_ref_sm)
y_pred_ref_test  = model_ref.predict(X_test_ref_sm)

r2_ref_train = r2_score(y_train, y_pred_ref_train)
r2_ref_test  = r2_score(y_test,  y_pred_ref_test)
rmse_ref = np.sqrt(mean_squared_error(y_test, y_pred_ref_test))
mae_ref  = mean_absolute_error(y_test, y_pred_ref_test)

print(f'R2 Train:  {r2_ref_train:.4f}')
print(f'R2 Test:   {r2_ref_test:.4f}')
print(f'RMSE Test: {rmse_ref:.4f}')
print(f'MAE Test:  {mae_ref:.4f}')

res_ref = y_test.values - y_pred_ref_test

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].scatter(y_pred_ref_test, res_ref, alpha=0.2, s=8, color='steelblue')
axes[0,0].axhline(0, color='red', linewidth=1)
axes[0,0].set_xlabel('Valores ajustados')
axes[0,0].set_ylabel('Residuos')
axes[0,0].set_title('Residuos vs Ajustados')

stats.probplot(res_ref, dist='norm', plot=axes[0,1])
axes[0,1].set_title('Q-Q Plot de Residuos')

axes[1,0].hist(res_ref, bins=60, color='salmon', edgecolor='white')
axes[1,0].set_xlabel('Residuo')
axes[1,0].set_ylabel('Frecuencia')
axes[1,0].set_title('Distribución de Residuos')

axes[1,1].scatter(y_test, y_pred_ref_test, alpha=0.2, s=8, color='steelblue')
axes[1,1].plot([float(y_test.min()), float(y_test.max())],
               [float(y_test.min()), float(y_test.max())], 'r--', linewidth=1.5)
axes[1,1].set_xlabel('Real')
axes[1,1].set_ylabel('Predicho')
axes[1,1].set_title('Real vs Predicho – Modelo Refinado')

plt.tight_layout()
plt.show()

### Interpretación del modelo refinado

Al eliminar variables con multicolinealidad alta o baja significancia estadística, el modelo gana en interpretabilidad. El Q-Q plot indica si los residuos siguen una distribución normal (condición deseable para inferencia).

## 8. Regularización: Ridge, Lasso y ElasticNet

- **Ridge (L2)**: reduce coeficientes pero no los anula.
- **Lasso (L1)**: puede llevar coeficientes a 0 (selección de variables).
- **ElasticNet (L1+L2)**: combinación de ambas.

In [ ]:
scaler_reg = StandardScaler()
X_train_sc = scaler_reg.fit_transform(X_train.astype(float))
X_test_sc  = scaler_reg.transform(X_test.astype(float))

alphas = np.logspace(-3, 3, 50)

model_ridge = RidgeCV(alphas=alphas, cv=5)
model_ridge.fit(X_train_sc, y_train)
print(f'Ridge     alpha óptimo: {model_ridge.alpha_:.4f}')

model_lasso = LassoCV(alphas=alphas, cv=5, random_state=RANDOM_STATE, max_iter=5000)
model_lasso.fit(X_train_sc, y_train)
print(f'Lasso     alpha óptimo: {model_lasso.alpha_:.4f}  coef=0: {np.sum(model_lasso.coef_==0)}/{len(model_lasso.coef_)}')

model_en = ElasticNetCV(alphas=alphas, l1_ratio=[0.1,0.5,0.7,0.9,0.95,1.0],
                        cv=5, random_state=RANDOM_STATE, max_iter=5000)
model_en.fit(X_train_sc, y_train)
print(f'ElasticNet alpha óptimo: {model_en.alpha_:.4f}  l1_ratio: {model_en.l1_ratio_:.2f}')

In [ ]:
coef_df = pd.DataFrame({
    'Feature':    feature_candidates,
    'Ridge':      model_ridge.coef_,
    'Lasso':      model_lasso.coef_,
    'ElasticNet': model_en.coef_
})

coef_df.set_index('Feature').plot(kind='bar', figsize=(14, 5))
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Coeficientes estandarizados: Ridge vs Lasso vs ElasticNet')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))

## 9. Evaluación y Selección del Mejor Modelo de Regresión Lineal

In [ ]:
def metrics(y_true, y_pred, label):
    return {
        'Modelo': label,
        'R2 Test': round(r2_score(y_true, y_pred), 4),
        'RMSE':    round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE':     round(mean_absolute_error(y_true, y_pred), 4)
    }

results = [
    metrics(y_test, y_pred_uni_test,                    'Univariado'),
    metrics(y_test, y_pred_multi_test,                  'Multivariado'),
    metrics(y_test, y_pred_ref_test,                    'Refinado'),
    metrics(y_test, model_ridge.predict(X_test_sc),     'Ridge'),
    metrics(y_test, model_lasso.predict(X_test_sc),     'Lasso'),
    metrics(y_test, model_en.predict(X_test_sc),        'ElasticNet'),
]

results_df = pd.DataFrame(results).sort_values('R2 Test', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_bar = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
axes[0].bar(results_df['Modelo'], results_df['R2 Test'], color=colors_bar, edgecolor='white')
axes[0].set_title('R2 en Test por Modelo')
axes[0].set_ylabel('R2')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=30)
for i, row in results_df.iterrows():
    axes[0].text(i, row['R2 Test'] + 0.01, f"{row['R2 Test']:.3f}", ha='center', fontsize=9)

axes[1].bar(results_df['Modelo'], results_df['RMSE'], color='salmon', edgecolor='white')
axes[1].set_title('RMSE en Test por Modelo')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Guardar métricas del mejor modelo lineal para comparación posterior
best_lineal_name = results_df.iloc[0]['Modelo']
best_lineal_r2   = results_df.iloc[0]['R2 Test']
best_lineal_rmse = results_df.iloc[0]['RMSE']
best_lineal_mae  = results_df.iloc[0]['MAE']

print(f'Mejor modelo lineal: {best_lineal_name}')
print(results_df.iloc[0].to_string())

best_preds_map = {
    'Univariado':   y_pred_uni_test,
    'Multivariado': y_pred_multi_test,
    'Refinado':     y_pred_ref_test,
    'Ridge':        model_ridge.predict(X_test_sc),
    'Lasso':        model_lasso.predict(X_test_sc),
    'ElasticNet':   model_en.predict(X_test_sc),
}
best_preds = best_preds_map[best_lineal_name]

plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_preds, alpha=0.2, s=8, color='steelblue')
mn, mx = float(y_test.min()), float(y_test.max())
plt.plot([mn,mx],[mn,mx], 'r--', linewidth=1.5, label='Predicción perfecta')
plt.xlabel('Precio real (log)')
plt.ylabel('Precio predicho (log)')
plt.title(f'Real vs Predicho – {best_lineal_name}')
plt.legend()
plt.tight_layout()
plt.show()

---
# PARTE II — ÁRBOLES DE DECISIÓN (Lab 4)
---

## 10. Árbol de Regresión – Modelo Base

Entrenamos un árbol de regresión sin restricción de profundidad. Usamos los **mismos** conjuntos train/test del Lab 3.

In [ ]:
tree_reg_full = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree_reg_full.fit(X_train, y_train)

y_pred_train_full = tree_reg_full.predict(X_train)
y_pred_test_full  = tree_reg_full.predict(X_test)

print('=== Árbol de Regresión sin restricción ===')
print(f'Profundidad real:  {tree_reg_full.get_depth()}')
print(f'Número de hojas:   {tree_reg_full.get_n_leaves()}')
print(f'R² Train:          {r2_score(y_train, y_pred_train_full):.4f}')
print(f'R² Test:           {r2_score(y_test,  y_pred_test_full):.4f}')
print(f'RMSE Test:         {np.sqrt(mean_squared_error(y_test, y_pred_test_full)):.4f}')
print(f'MAE Test:          {mean_absolute_error(y_test, y_pred_test_full):.4f}')

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_test_full, alpha=0.2, s=8, color='steelblue')
mn, mx = float(y_test.min()), float(y_test.max())
plt.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Predicción perfecta')
plt.xlabel('log(Precio) real')
plt.ylabel('log(Precio) predicho')
plt.title('Árbol de Regresión (sin restricción) – Test')
plt.legend()
plt.tight_layout()
plt.show()

### Análisis del modelo base

El árbol sin restricción memoriza completamente los datos de entrenamiento (R² ≈ 1.00 en train), pero generaliza peor en test — síntoma clásico de **sobreajuste**. La gran diferencia entre R² train y test indica que el árbol es demasiado profundo y se ha adaptado al ruido del conjunto de entrenamiento.

## 11. Tuneo de Profundidad – Árbol de Regresión

Evaluamos al menos 3 modelos adicionales con distinta profundidad máxima.

In [ ]:
depths = [3, 5, 7, 10, 15, None]
reg_results = []

for d in depths:
    model = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    y_tr = model.predict(X_train)
    y_te = model.predict(X_test)
    reg_results.append({
        'max_depth': str(d) if d else 'Sin límite',
        'R2_train':  round(r2_score(y_train, y_tr), 4),
        'R2_test':   round(r2_score(y_test,  y_te), 4),
        'RMSE':      round(np.sqrt(mean_squared_error(y_test, y_te)), 4),
        'MAE':       round(mean_absolute_error(y_test, y_te), 4),
        '_model':    model
    })

reg_df = pd.DataFrame(reg_results)
print(reg_df.drop(columns='_model').to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_pos = range(len(reg_df))

axes[0].plot(x_pos, reg_df['R2_train'], 'o-', label='R² Train', color='salmon')
axes[0].plot(x_pos, reg_df['R2_test'],  's-', label='R² Test',  color='steelblue')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(reg_df['max_depth'])
axes[0].set_xlabel('Profundidad máxima')
axes[0].set_ylabel('R²')
axes[0].set_title('R² por profundidad – Árbol de Regresión')
axes[0].legend()

axes[1].plot(x_pos, reg_df['RMSE'], 'o-', color='darkorange', label='RMSE')
axes[1].plot(x_pos, reg_df['MAE'],  's-', color='green',      label='MAE')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(reg_df['max_depth'])
axes[1].set_xlabel('Profundidad máxima')
axes[1].set_ylabel('Error')
axes[1].set_title('RMSE/MAE por profundidad – Árbol de Regresión')
axes[1].legend()

plt.tight_layout()
plt.show()

best_reg_idx   = reg_df['R2_test'].idxmax()
best_reg_row   = reg_df.iloc[best_reg_idx]
best_reg_model = reg_df.loc[best_reg_idx, '_model']

print(f'Mejor profundidad: {best_reg_row["max_depth"]}')
print(f'R² Test:  {best_reg_row["R2_test"]}')
print(f'RMSE:     {best_reg_row["RMSE"]}')
print(f'GAP:      {best_reg_row["R2_train"] - best_reg_row["R2_test"]:.4f}')

### Interpretación del tuneo

A medida que aumenta la profundidad, el R² de entrenamiento sube hasta 1.0, pero el R² de prueba alcanza un máximo y luego se deteriora (sobreajuste). El árbol con profundidad intermedia equilibra mejor sesgo y varianza.

In [ ]:
# Importancia de variables del mejor árbol de regresión
feat_imp = pd.Series(best_reg_model.feature_importances_, index=feature_candidates)
feat_imp = feat_imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='steelblue', edgecolor='white')
plt.gca().invert_yaxis()
plt.xlabel('Importancia (MSE)')
plt.title(f'Top 15 variables – Árbol de Regresión (depth={best_reg_row["max_depth"]})')
plt.tight_layout()
plt.show()

## 12. Comparación: Árbol de Regresión vs Regresión Lineal

In [ ]:
comp_lineal_vs_arbol = pd.DataFrame([
    {'Modelo': f'{best_lineal_name} (Reg. Lineal)',
     'R² Test': best_lineal_r2,         'RMSE': best_lineal_rmse, 'MAE': best_lineal_mae},
    {'Modelo': f'Árbol Reg. (depth={best_reg_row["max_depth"]})',
     'R² Test': best_reg_row['R2_test'], 'RMSE': best_reg_row['RMSE'], 'MAE': best_reg_row['MAE']},
])
print(comp_lineal_vs_arbol.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(2)
modelos = comp_lineal_vs_arbol['Modelo']

axes[0].bar(x, comp_lineal_vs_arbol['R² Test'], color=['steelblue','darkorange'], edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(modelos, rotation=15, ha='right')
axes[0].set_ylabel('R² Test')
axes[0].set_title('R² Test: Reg. Lineal vs Árbol')
axes[0].set_ylim(0, 1)

axes[1].bar(x, comp_lineal_vs_arbol['RMSE'], color=['steelblue','darkorange'], edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(modelos, rotation=15, ha='right')
axes[1].set_ylabel('RMSE Test')
axes[1].set_title('RMSE Test (menor = mejor)')

plt.tight_layout()
plt.show()

### Interpretación

| Modelo | Ventaja |
|---|---|
| Regresión Lineal | Interpretable, estable, buen desempeño cuando las relaciones son lineales |
| Árbol de Regresión | Captura no linealidades, no requiere supuestos distribucionales |

Si el árbol supera a la regresión lineal en R² y RMSE en test, confirma que el precio de Airbnb tiene relaciones **no lineales** que el modelo lineal no captura.

## 13. Creación de la Variable Categórica: Económicas / Intermedias / Caras

La variable `precio_categoria` se construye sobre el **precio original por noche** (sin log). Los límites se definen usando los **percentiles 33 y 67** del conjunto de entrenamiento.

In [ ]:
print('Estadísticas del precio por noche:')
print(price_raw.describe().round(2))
print()
for p in [10, 25, 33, 50, 67, 75, 90, 95]:
    print(f'  P{p:2d}: ${price_raw.quantile(p/100):.2f}')

In [ ]:
# Límites calculados SOLO en train (evita fuga de información)
p33_train = price_train.quantile(0.33)
p67_train = price_train.quantile(0.67)

print(f'Límite Económica  → Intermedia : ${p33_train:.2f}  (P33 del train)')
print(f'Límite Intermedia → Cara       : ${p67_train:.2f}  (P67 del train)')
print()
print(f'  Económica  : precio ≤ ${p33_train:.2f}')
print(f'  Intermedia : ${p33_train:.2f} < precio ≤ ${p67_train:.2f}')
print(f'  Cara       : precio > ${p67_train:.2f}')

In [ ]:
def categorizar_precio(series, lim_bajo, lim_alto):
    return pd.cut(series,
                  bins=[-np.inf, lim_bajo, lim_alto, np.inf],
                  labels=['Economica', 'Intermedia', 'Cara'])

y_cat_train = categorizar_precio(price_train, p33_train, p67_train)
y_cat_test  = categorizar_precio(price_test,  p33_train, p67_train)

print('Distribución TRAIN:')
print(y_cat_train.value_counts().sort_index())
print(f'  (proporciones: {y_cat_train.value_counts(normalize=True).sort_index().round(3).to_dict()})')
print('\nDistribución TEST:')
print(y_cat_test.value_counts().sort_index())

In [ ]:
plt.figure(figsize=(12, 5))
plt.hist(price_raw, bins=100, color='lightgray', edgecolor='white', label='Precio')
plt.axvline(p33_train, color='steelblue', linestyle='--', linewidth=2,
            label=f'P33 = ${p33_train:.0f}')
plt.axvline(p67_train, color='darkorange', linestyle='--', linewidth=2,
            label=f'P67 = ${p67_train:.0f}')
plt.axvspan(0,           p33_train,              alpha=0.12, color='green', label='Económica')
plt.axvspan(p33_train,   p67_train,              alpha=0.12, color='blue',  label='Intermedia')
plt.axvspan(p67_train,   price_raw.max()*1.05,   alpha=0.12, color='red',   label='Cara')
plt.xlabel('Precio por noche (USD)')
plt.ylabel('Frecuencia')
plt.title('Distribución del precio y límites de categorías')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

### Justificación de los límites

1. **Balance de clases**: los terciles garantizan ~33% de los datos en cada categoría, evitando clases muy desbalanceadas.
2. **Base estadística**: los límites reflejan la distribución real del mercado Airbnb, no valores arbitrarios.
3. **Sin fuga de información**: los percentiles se calculan únicamente con el training set.
4. **Interpretabilidad de negocio**: las categorías representan segmentos claros del mercado para SmartStay.

## 14. Árbol de Clasificación – Modelo Base

Entrenamos con `precio_categoria`. **El precio original NO se incluye** como feature — causaría fuga de información perfecta.

In [ ]:
tree_clf_full = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree_clf_full.fit(X_train, y_cat_train)

y_pred_clf_train = tree_clf_full.predict(X_train)
y_pred_clf_test  = tree_clf_full.predict(X_test)

acc_train = accuracy_score(y_cat_train, y_pred_clf_train)
acc_test  = accuracy_score(y_cat_test,  y_pred_clf_test)

print('=== Árbol de Clasificación sin restricción ===')
print(f'Profundidad real:  {tree_clf_full.get_depth()}')
print(f'Número de hojas:   {tree_clf_full.get_n_leaves()}')
print(f'Accuracy Train:    {acc_train:.4f}')
print(f'Accuracy Test:     {acc_test:.4f}')

In [ ]:
# Visualización gráfica del árbol (depth=4 para legibilidad)
tree_clf_viz = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
tree_clf_viz.fit(X_train, y_cat_train)

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    tree_clf_viz,
    feature_names=feature_candidates,
    class_names=['Cara', 'Economica', 'Intermedia'],
    filled=True, rounded=True, fontsize=8, ax=ax
)
ax.set_title('Árbol de Clasificación (max_depth=4) – Precio Categoría', fontsize=14)
plt.tight_layout()
plt.savefig('tree_clasificacion_viz.png', dpi=120, bbox_inches='tight')
plt.show()
print('Árbol guardado como tree_clasificacion_viz.png')

In [ ]:
# Reglas del árbol en texto (depth=3)
tree_clf_rules = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
tree_clf_rules.fit(X_train, y_cat_train)
print('Reglas del árbol (depth=3):')
print(export_text(tree_clf_rules, feature_names=feature_candidates))

### Interpretación del árbol de clasificación

El árbol aprende reglas jerárquicas para clasificar el precio. Las variables en los nodos superiores son las más discriminantes: típicamente `accommodates` (capacidad) y las dummies de `room_type` aparecen primero, confirmando que el **tamaño y tipo de propiedad** son los factores más determinantes del segmento de precio en el mercado Airbnb.

## 15. Evaluación en Conjunto de Prueba

In [ ]:
print('=== Evaluación en Test – Árbol sin restricción ===')
print(f'Accuracy: {acc_test:.4f}')
print()
print(classification_report(y_cat_test, y_pred_clf_test,
                             target_names=['Economica', 'Intermedia', 'Cara']))

## 16. Matriz de Confusión – Árbol de Clasificación

In [ ]:
classes = ['Economica', 'Intermedia', 'Cara']
cm = confusion_matrix(y_cat_test, y_pred_clf_test, labels=classes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Árbol – Matriz de Confusión (absoluta)')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm.round(2), display_labels=classes).plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('Árbol – Matriz de Confusión (normalizada)')

plt.tight_layout()
plt.show()

print('Análisis de errores:')
for i, c in enumerate(classes):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    print(f'  {c:12s} → correctos: {tp:,}, errores: {fn:,}, precisión fila: {cm_norm[i,i]:.2%}')

### Análisis de la matriz de confusión

- **Donde acierta más**: categorías extremas (Económica y Cara), cuyos precios están más alejados del centro.
- **Donde se equivoca más**: categoría **Intermedia** — los precios fronterizos generan confusión con las categorías vecinas.
- **Importancia de los errores**: desde el punto de vista de SmartStay, clasificar una propiedad *Cara* como *Económica* tiene mayor impacto operativo que confundir *Intermedia* con cualquiera de las dos.

## 17. Validación Cruzada

In [ ]:
DEPTH_CV = 7

tree_cv = DecisionTreeClassifier(max_depth=DEPTH_CV, random_state=RANDOM_STATE)
cv_scores = cross_val_score(tree_cv, X_train, y_cat_train, cv=5, scoring='accuracy')

print(f'Validación cruzada 5-fold (depth={DEPTH_CV}):')
print(f'  Scores: {np.round(cv_scores, 4)}')
print(f'  Media:  {cv_scores.mean():.4f}')
print(f'  Std:    {cv_scores.std():.4f}')

tree_cv.fit(X_train, y_cat_train)
y_pred_cv_test = tree_cv.predict(X_test)
acc_cv_test = accuracy_score(y_cat_test, y_pred_cv_test)

print(f'\nAccuracy en Test (árbol CV depth={DEPTH_CV}): {acc_cv_test:.4f}')
print(f'Accuracy árbol sin restricción:               {acc_test:.4f}')
print(f'Diferencia: {acc_cv_test - acc_test:+.4f}')

## 18. Tuneo de Profundidad – Árbol de Clasificación

In [ ]:
depths_clf = [3, 5, 7, 10, 15, None]
clf_results = []

for d in depths_clf:
    model = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    model.fit(X_train, y_cat_train)
    acc_tr = accuracy_score(y_cat_train, model.predict(X_train))
    acc_te = accuracy_score(y_cat_test,  model.predict(X_test))
    cv_sc  = cross_val_score(model, X_train, y_cat_train, cv=5, scoring='accuracy').mean()
    clf_results.append({
        'max_depth': str(d) if d else 'Sin límite',
        'Acc_train': round(acc_tr, 4),
        'Acc_test':  round(acc_te, 4),
        'CV_mean':   round(cv_sc, 4),
        '_model':    model
    })

clf_df = pd.DataFrame(clf_results)
print(clf_df.drop(columns='_model').to_string(index=False))

In [ ]:
x_pos = range(len(clf_df))

plt.figure(figsize=(10, 5))
plt.plot(x_pos, clf_df['Acc_train'], 'o-', label='Accuracy Train', color='salmon')
plt.plot(x_pos, clf_df['Acc_test'],  's-', label='Accuracy Test',  color='steelblue')
plt.plot(x_pos, clf_df['CV_mean'],   '^-', label='CV Mean (5-fold)', color='green')
plt.xticks(x_pos, clf_df['max_depth'])
plt.xlabel('Profundidad máxima')
plt.ylabel('Accuracy')
plt.title('Accuracy por profundidad – Árbol de Clasificación')
plt.legend()
plt.tight_layout()
plt.show()

best_clf_idx   = clf_df['Acc_test'].idxmax()
best_clf_row   = clf_df.iloc[best_clf_idx]
best_clf_model = clf_df.loc[best_clf_idx, '_model']
print(f'Mejor árbol clasificación: depth={best_clf_row["max_depth"]}, Acc_test={best_clf_row["Acc_test"]}')

### Interpretación del tuneo del árbol de clasificación

El árbol sin límite memoriza el training set (accuracy ≈ 1.0) pero generaliza peor. La validación cruzada ofrece una estimación más robusta y ayuda a elegir la profundidad que maximiza la generalización.

## 19. Random Forest

Al promediar múltiples árboles entrenados en subconjuntos aleatorios de datos y features, el Random Forest reduce la varianza sin aumentar el sesgo. Entrenamos modelos para **regresión** y **clasificación**.

In [ ]:
# --- Random Forest Regresión ---
rf_reg = RandomForestRegressor(
    n_estimators=200, min_samples_leaf=5,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_reg.fit(X_train, y_train)

r2_rf_test   = r2_score(y_test,  rf_reg.predict(X_test))
rmse_rf_test = np.sqrt(mean_squared_error(y_test, rf_reg.predict(X_test)))
mae_rf_test  = mean_absolute_error(y_test, rf_reg.predict(X_test))

print('=== Random Forest – Regresión (200 árboles) ===')
print(f'R² Train:  {r2_score(y_train, rf_reg.predict(X_train)):.4f}')
print(f'R² Test:   {r2_rf_test:.4f}')
print(f'RMSE Test: {rmse_rf_test:.4f}')
print(f'MAE Test:  {mae_rf_test:.4f}')

In [ ]:
# --- Random Forest Clasificación ---
rf_clf = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=5,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_clf.fit(X_train, y_cat_train)

y_pred_rf_clf_test = rf_clf.predict(X_test)
acc_rf_test = accuracy_score(y_cat_test, y_pred_rf_clf_test)

print('=== Random Forest – Clasificación (200 árboles) ===')
print(f'Accuracy Train: {accuracy_score(y_cat_train, rf_clf.predict(X_train)):.4f}')
print(f'Accuracy Test:  {acc_rf_test:.4f}')
print()
print(classification_report(y_cat_test, y_pred_rf_clf_test,
                             target_names=['Economica', 'Intermedia', 'Cara']))

In [ ]:
# Matrices de confusión RF
cm_rf = confusion_matrix(y_cat_test, y_pred_rf_clf_test, labels=classes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(cm_rf, display_labels=classes).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Random Forest – Matriz de Confusión (absoluta)')

cm_rf_norm = cm_rf.astype(float) / cm_rf.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_rf_norm.round(2), display_labels=classes).plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Random Forest – Matriz de Confusión (normalizada)')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de variables RF
rf_feat_imp = pd.Series(rf_clf.feature_importances_, index=feature_candidates)
rf_feat_imp = rf_feat_imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
rf_feat_imp.plot(kind='barh', color='darkorange', edgecolor='white')
plt.gca().invert_yaxis()
plt.xlabel('Importancia (Gini promediado)')
plt.title('Top 15 variables más importantes – Random Forest Clasificación')
plt.tight_layout()
plt.show()

## 20. Comparación Final de Todos los Algoritmos

In [ ]:
print('='*65)
print('COMPARACIÓN – MODELOS DE REGRESIÓN (target: log_price)')
print('='*65)

comp_reg = pd.DataFrame([
    {'Modelo': f'{best_lineal_name} (Reg. Lineal)',
     'R² Test': best_lineal_r2,         'RMSE': best_lineal_rmse, 'MAE': best_lineal_mae},
    {'Modelo': f'Árbol Reg. (depth={best_reg_row["max_depth"]})',
     'R² Test': best_reg_row['R2_test'], 'RMSE': best_reg_row['RMSE'], 'MAE': best_reg_row['MAE']},
    {'Modelo': 'Random Forest Reg. (200 árboles)',
     'R² Test': round(r2_rf_test, 4),   'RMSE': round(rmse_rf_test, 4), 'MAE': round(mae_rf_test, 4)},
]).sort_values('R² Test', ascending=False).reset_index(drop=True)

print(comp_reg.to_string(index=False))
print(f'\nGanador regresión: {comp_reg.iloc[0]["Modelo"]}')

print()
print('='*65)
print('COMPARACIÓN – MODELOS DE CLASIFICACIÓN')
print('='*65)

comp_clf = pd.DataFrame([
    {'Modelo': 'Árbol sin restricción',             'Accuracy Test': round(acc_test, 4)},
    {'Modelo': f'Árbol (depth={best_clf_row["max_depth"]})',
                                                     'Accuracy Test': best_clf_row['Acc_test']},
    {'Modelo': f'Árbol + CV (depth={DEPTH_CV})',     'Accuracy Test': round(acc_cv_test, 4)},
    {'Modelo': 'Random Forest Clf. (200 árboles)',   'Accuracy Test': round(acc_rf_test, 4)},
]).sort_values('Accuracy Test', ascending=False).reset_index(drop=True)

print(comp_clf.to_string(index=False))
print(f'\nGanador clasificación: {comp_clf.iloc[0]["Modelo"]}')

In [ ]:
# Gráficos comparativos finales
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modelos_wrap  = comp_reg['Modelo'].str.wrap(20)
colors_reg    = ['gold' if i == 0 else 'steelblue' for i in range(len(comp_reg))]

axes[0].barh(modelos_wrap, comp_reg['R² Test'], color=colors_reg, edgecolor='white')
axes[0].set_xlabel('R² Test')
axes[0].set_title('R² Test – Modelos de Regresión')
axes[0].invert_yaxis()

axes[1].barh(modelos_wrap, comp_reg['RMSE'], color=colors_reg, edgecolor='white')
axes[1].set_xlabel('RMSE Test')
axes[1].set_title('RMSE Test – Modelos de Regresión (menor = mejor)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
colors_clf = ['gold' if i == 0 else 'steelblue' for i in range(len(comp_clf))]
plt.barh(comp_clf['Modelo'].str.wrap(25), comp_clf['Accuracy Test'],
         color=colors_clf, edgecolor='white')
plt.xlabel('Accuracy Test')
plt.title('Accuracy Test – Modelos de Clasificación')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 21. Conclusiones Generales

### Regresión (predicción del precio por noche)

1. **Regresión Lineal**: establece el baseline. Ridge y Lasso controlan el overfitting, pero no capturan relaciones no lineales.
2. **Árbol sin restricción**: sobreajuste severo (R² train ≈ 1.0). Al limitar la profundidad se reduce el gap train/test y mejora la generalización.
3. **Random Forest**: promedia 200 árboles, reduciendo la varianza. Es el modelo más preciso de los tres grupos.

### Clasificación (Económica / Intermedia / Cara)

1. **Variable categórica**: percentiles 33/67 del training set — clases balanceadas, base estadística sólida, sin fuga de información.
2. **Árbol de clasificación**: interpreta bien los extremos, confunde más la zona Intermedia.
3. **Validación cruzada**: estabiliza la selección de profundidad óptima.
4. **Random Forest**: mayor accuracy gracias al ensamblado que reduce la varianza en fronteras de clase.

### Recomendación para SmartStay Advisors

| Tarea | Modelo recomendado | Razón |
|---|---|---|
| Estimar precio exacto | **Random Forest Regresor** | Mayor R², menor RMSE |
| Segmentar propiedades | **Random Forest Clasificador** | Mayor accuracy, robusto a outliers |
| Explicar a clientes | **Árbol depth=4** | Visualizable e interpretable |